# AgriSmart Assistant: Domain-Specific Agricultural QA via LLM Fine-Tuning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kellenmurerwa/AgriSmart-Assistant/blob/main/AgriSmart_Assistant.ipynb)

This notebook implements a domain-specific agricultural assistant by fine-tuning **TinyLlama-1.1B-Chat** using **LoRA (Low-Rank Adaptation)** on the **KisanVaani Agriculture QA** dataset. The fine-tuned model can answer questions about crops, pests, soil management, and fertilizers.

**Pipeline Overview:**
1. Dataset collection and preprocessing
2. Base model loading with 4-bit quantization
3. LoRA (PEFT) fine-tuning with two hyperparameter experiments
4. Evaluation (BLEU, ROUGE, Perplexity)
5. Base model vs fine-tuned comparison
6. Gradio chatbot deployment

## 1. Project Definition & Domain Alignment

**Domain:** Agriculture

**Problem Statement:** Smallholder farmers and agricultural workers often lack access to timely, expert agricultural advice. While general-purpose LLMs can provide some guidance, they frequently lack the domain-specific knowledge needed to answer detailed questions about crop diseases, pest management, soil health, and fertilizer usage.

**Objective:** Fine-tune a lightweight LLM to serve as a specialized agricultural assistant that provides accurate, context-aware responses to farming-related queries.

**Why Fine-Tuning?**
- General LLMs lack specialized agricultural vocabulary and domain knowledge
- Fine-tuning on curated agricultural QA data improves response relevance and accuracy
- Parameter-efficient fine-tuning (LoRA) enables training on consumer GPUs
- A domain-specific model is more reliable than prompting a general model

**Model Choice:** TinyLlama-1.1B-Chat-v1.0 â€” a compact yet capable generative model that balances performance with practical training constraints on free GPU resources (Colab T4 / consumer GPUs).

## 2. Install Dependencies

In [1]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl evaluate gradio rouge_score nltk absl-py

## 3. Import Libraries

In [ ]:
import torch
import pandas as pd
import time
import gc
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTConfig, SFTTrainer
from evaluate import load
import gradio as gr

# === Device setup (used by all cells below) ===
device = "cuda" if torch.cuda.is_available() else "cpu"

# Check GPU availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    gpu_props = torch.cuda.get_device_properties(0)
    gpu_mem = getattr(gpu_props, 'total_memory', None) or getattr(gpu_props, 'total_mem', None)
    if gpu_mem:
        print(f"GPU Memory: {gpu_mem / 1024**3:.1f} GB")

# Auto-detect precision
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    compute_dtype = torch.bfloat16
    use_bf16 = True
    print("Using bfloat16 precision (Ampere+ GPU detected)")
else:
    compute_dtype = torch.float16
    use_bf16 = False
    print("Using float16 precision")

## Quick Start: Load Pre-trained Model (Skip Training)

**Want to test the model without retraining?** Run the cell below to load the fine-tuned LoRA adapter directly from Hugging Face, then skip to **Section 11 (Gradio Deployment)** to launch the chatbot.

This loads the best model (Experiment 2) from: [kellenmurerwa/AgriSmart-TinyLlama-LoRA](https://huggingface.co/kellenmurerwa/AgriSmart-TinyLlama-LoRA)

In [ ]:
# === QUICK START: Load pre-trained model from Hugging Face ===
# Run this cell INSTEAD of Sections 4-10 to skip training and go straight to the chatbot.
# After running this cell, jump to Section 11 (Gradio Deployment).

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("kellenmurerwa/AgriSmart-TinyLlama-LoRA")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load base model (4-bit quantization on GPU, full precision on CPU)
if device == "cuda":
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto"
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        device_map="cpu"
    )

# Load fine-tuned LoRA adapter from Hugging Face
model = PeftModel.from_pretrained(base_model, "kellenmurerwa/AgriSmart-TinyLlama-LoRA")

print("Fine-tuned AgriSmart model loaded from Hugging Face!")
if device == "cuda":
    print(f"GPU memory used: {torch.cuda.memory_allocated() / 1024**2:.0f} MB")
else:
    print("Running on CPU (no GPU detected). Inference will be slower.")
print("\n>>> Now skip to Section 11 (Gradio Deployment) to launch the chatbot.")

## 4. Dataset Collection & Preprocessing

We use the **KisanVaani Agriculture QA** dataset from Hugging Face, which contains question-answer pairs about agriculture, covering topics like crop management, pest control, soil health, and fertilizer usage.

**Dataset:** [KisanVaani/agriculture-qa-english-only](https://huggingface.co/datasets/KisanVaani/agriculture-qa-english-only)

### 4.1 Load Dataset

In [3]:
dataset = load_dataset("KisanVaani/agriculture-qa-english-only", split="train")
print(f"Total samples: {len(dataset)}")
print(f"\nColumns: {dataset.column_names}")
print(f"\nSample entry:")
print(f"  Question: {dataset[0]['question']}")
print(f"  Answer: {dataset[0]['answers'][:200]}...")

Total samples: 22615

Columns: ['question', 'answers']

Sample entry:
  Question: What are the recommended varieties of Wheat?
  Answer: The recommended varieties of Wheat include HD 2967, HD 3086, WH 1105, PBW 550, DBW 17, HD 2851, UP 2338, PBW 343, WH 542, RAJ 3765, K 9107, NW 1014, and NW 2036. These varieties are kn...


### 4.2 Data Cleaning

Remove entries with null questions or answers to ensure data quality.

In [4]:
# Remove null entries
dataset = dataset.filter(lambda x: x["question"] is not None and x["answers"] is not None)
print(f"Samples after cleaning: {len(dataset)}")

Samples after cleaning: 22615


### 4.3 Format into Instruction-Response Template

We format each example into a clear instruction-response template that the model will learn to follow. This structured format helps the model understand when to generate a response.

In [5]:
def format_prompt(example):
    """Format each QA pair into an instruction-response template."""
    return {
        "text": f"""### Instruction:\n{example['question']}\n\n### Response:\n{example['answers']}"""
    }

dataset = dataset.map(format_prompt)

# Preview a formatted example
print("Formatted example:")
print(dataset[0]["text"][:300])

Formatted example:
### Instruction:
What are the recommended varieties of Wheat?

### Response:
The recommended varieties of Wheat include HD 2967, HD 3086, WH 1105, PBW 550, DBW 17, HD 2851, UP 2338, PBW 343, WH 542, RAJ 3765, K 9107, NW 1014, and NW 2036. These varieties are known for their high yield potential and suitability for different agro-cl


### 4.4 Train-Test Split

Split the dataset into 90% training and 10% evaluation sets with a fixed seed for reproducibility.

In [6]:
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print(f"Training samples: {len(train_dataset)}")
print(f"Evaluation samples: {len(eval_dataset)}")

Training samples: 20353
Evaluation samples: 2262


## 5. Load Base Model with 4-bit Quantization

We load **TinyLlama-1.1B-Chat-v1.0** with 4-bit NF4 quantization using `bitsandbytes`. This reduces memory usage from ~4.4GB to ~700MB, making it feasible to fine-tune on free GPU resources.

**Quantization config:**
- 4-bit NF4 quantization with double quantization
- Compute dtype auto-detected (bfloat16 for Ampere+, float16 for older GPUs)

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load base model (4-bit quantization on GPU, full precision on CPU)
if device == "cuda":
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto"
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        device_map="cpu"
    )

print(f"Model loaded: {model_name}")
print(f"Model parameters: {model.num_parameters():,}")
if device == "cuda":
    print(f"GPU memory used: {torch.cuda.memory_allocated() / 1024**2:.0f} MB")
else:
    print("Running on CPU")

## 6. Apply LoRA (Parameter-Efficient Fine-Tuning)

We apply **LoRA (Low-Rank Adaptation)** to fine-tune only a small fraction of the model's parameters. This makes training fast and memory-efficient while preserving the model's general capabilities.

**LoRA Configuration:**
- Rank (r): 16
- Alpha: 32
- Target modules: q_proj, v_proj (attention layers)
- Dropout: 0.05

In [8]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


## 7. Hyperparameter Experiments

We conduct two training experiments with different hyperparameter configurations to find the optimal setup. We track training loss, accuracy, time, and GPU memory usage for comparison.

### Experiment 1
- **Learning Rate:** 2e-4 (higher, for faster initial convergence)
- **Batch Size:** 2 (with gradient accumulation of 4 = effective batch of 8)
- **Epochs:** 1

In [ ]:
# Reset peak memory counter
if device == "cuda":
    torch.cuda.reset_peak_memory_stats()

training_config_1 = SFTConfig(
    output_dir="./agri_exp1",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    bf16=use_bf16 and device == "cuda",
    fp16=(not use_bf16) and device == "cuda",
    logging_steps=50,
    save_strategy="epoch",
    max_seq_length=512,
    dataset_text_field="text",
    report_to="none",
)

trainer_1 = SFTTrainer(
    model=model,
    args=training_config_1,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print("Starting Experiment 1...")
exp1_start = time.time()
trainer_1.train()
exp1_time = time.time() - exp1_start
exp1_peak_mem = torch.cuda.max_memory_allocated() / 1024**2 if device == "cuda" else 0

# Get final metrics from training log
exp1_final_loss = trainer_1.state.log_history[-1].get("loss", trainer_1.state.log_history[-2].get("loss", "N/A"))
exp1_final_acc = trainer_1.state.log_history[-1].get("mean_token_accuracy", trainer_1.state.log_history[-2].get("mean_token_accuracy", "N/A"))

print(f"\nExperiment 1 completed in {exp1_time:.1f} seconds")
print(f"Final Loss: {exp1_final_loss}")
print(f"Final Accuracy: {exp1_final_acc}")
if device == "cuda":
    print(f"Peak GPU Memory: {exp1_peak_mem:.0f} MB")

### Experiment 2
- **Learning Rate:** 5e-5 (lower, for more stable fine-grained learning)
- **Batch Size:** 4 (with gradient accumulation of 2 = effective batch of 8)
- **Epochs:** 2 (longer training for deeper adaptation)

In [ ]:
if device == "cuda":
    torch.cuda.reset_peak_memory_stats()

training_config_2 = SFTConfig(
    output_dir="./agri_exp2",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    num_train_epochs=2,
    bf16=use_bf16 and device == "cuda",
    fp16=(not use_bf16) and device == "cuda",
    logging_steps=50,
    save_strategy="epoch",
    max_seq_length=512,
    dataset_text_field="text",
    report_to="none",
)

trainer_2 = SFTTrainer(
    model=model,
    args=training_config_2,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print("Starting Experiment 2...")
exp2_start = time.time()
trainer_2.train()
exp2_time = time.time() - exp2_start
exp2_peak_mem = torch.cuda.max_memory_allocated() / 1024**2 if device == "cuda" else 0

exp2_final_loss = trainer_2.state.log_history[-1].get("loss", trainer_2.state.log_history[-2].get("loss", "N/A"))
exp2_final_acc = trainer_2.state.log_history[-1].get("mean_token_accuracy", trainer_2.state.log_history[-2].get("mean_token_accuracy", "N/A"))

print(f"\nExperiment 2 completed in {exp2_time:.1f} seconds")
print(f"Final Loss: {exp2_final_loss}")
print(f"Final Accuracy: {exp2_final_acc}")
if device == "cuda":
    print(f"Peak GPU Memory: {exp2_peak_mem:.0f} MB")

## 8. Experiment Comparison Table

The table below documents the impact of different hyperparameter configurations on training performance, including GPU memory usage and training time.

In [ ]:
comparison_data = {
    "Experiment": ["Exp 1", "Exp 2"],
    "Learning Rate": ["2e-4", "5e-5"],
    "Batch Size": [2, 4],
    "Grad Accum": [4, 2],
    "Effective Batch": [8, 8],
    "Epochs": [1, 2],
    "Total Steps": [trainer_1.state.global_step, trainer_2.state.global_step],
    "Final Loss": [round(exp1_final_loss, 4), round(exp2_final_loss, 4)],
    "Final Accuracy": [round(exp1_final_acc, 4), round(exp2_final_acc, 4)],
    "Training Time (s)": [round(exp1_time, 1), round(exp2_time, 1)],
}
if device == "cuda":
    comparison_data["Peak GPU Memory (MB)"] = [round(exp1_peak_mem), round(exp2_peak_mem)]

comparison_table = pd.DataFrame(comparison_data)

print("=" * 80)
print("HYPERPARAMETER EXPERIMENT COMPARISON")
print("=" * 80)
print(comparison_table.to_string(index=False))

print(f"\nAnalysis:")
print(f"- Experiment 2 achieves lower final loss ({round(exp2_final_loss, 4)} vs {round(exp1_final_loss, 4)})")
print(f"- Experiment 2 achieves higher accuracy ({round(exp2_final_acc, 4)} vs {round(exp1_final_acc, 4)})")
print(f"- The lower learning rate with more epochs leads to better convergence")
print(f"- Experiment 2 is selected as the best model for evaluation")

## 9. Evaluation

We evaluate the best model (Experiment 2) using standard NLP metrics and qualitative analysis.

### 9.1 BLEU & ROUGE Scores

We generate responses for 100 evaluation samples and compute:
- **BLEU**: Measures n-gram precision between generated and reference responses
- **ROUGE**: Measures recall-oriented overlap (ROUGE-1, ROUGE-2, ROUGE-L)

In [ ]:
bleu = load("bleu")
rouge = load("rouge")

predictions = []
references = []

eval_samples = min(100, len(eval_dataset))
print(f"Evaluating on {eval_samples} samples...\n")

for i, example in enumerate(eval_dataset.select(range(eval_samples))):
    prompt = f"""### Instruction:\n{example['question']}\n\n### Response:\n"""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=100, do_sample=False)

    decoded = tokenizer.decode(output[0], skip_special_tokens=True)

    # Extract only the generated response (after ### Response:)
    if "### Response:" in decoded:
        pred = decoded.split("### Response:")[-1].strip()
    else:
        pred = decoded.strip()

    predictions.append(pred)
    references.append(example["answers"])

    if (i + 1) % 20 == 0:
        print(f"  Evaluated {i + 1}/{eval_samples}")

# Compute metrics
print("\nComputing BLEU...")
bleu_score = bleu.compute(predictions=predictions, references=[[r] for r in references])

print("Computing ROUGE...")
rouge_score = rouge.compute(predictions=predictions, references=references)

print("\n" + "=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
print(f"BLEU Score:   {bleu_score['bleu']:.4f}")
print(f"ROUGE-1:      {rouge_score['rouge1']:.4f}")
print(f"ROUGE-2:      {rouge_score['rouge2']:.4f}")
print(f"ROUGE-L:      {rouge_score['rougeL']:.4f}")
print(f"ROUGE-Lsum:   {rouge_score['rougeLsum']:.4f}")

### 9.2 Perplexity

Perplexity measures how well the model predicts the evaluation data. Lower perplexity indicates better language modeling performance.

In [ ]:
def compute_perplexity(model, dataset, tokenizer, num_samples=100):
    """Compute perplexity on a subset of the dataset."""
    losses = []
    for example in dataset.select(range(min(num_samples, len(dataset)))):
        inputs = tokenizer(
            example["text"], return_tensors="pt",
            truncation=True, max_length=512
        ).to(device)
        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
        losses.append(outputs.loss.item())
    return torch.exp(torch.tensor(losses).mean()).item()

perplexity = compute_perplexity(model, eval_dataset, tokenizer)
print(f"Perplexity: {perplexity:.4f}")
print(f"(Lower is better - indicates how well the model predicts evaluation data)")

## 10. Base Model vs Fine-Tuned Comparison

We compare the base (pre-trained) TinyLlama model against our fine-tuned version to demonstrate the value of domain-specific fine-tuning. The same agricultural questions are asked to both models.

In [ ]:
# Free memory and load base model for comparison
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

if device == "cuda":
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto"
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        device_map="cpu"
    )

test_questions = [
    "How do I prevent cassava mosaic disease?",
    "What is the best fertilizer for rice crops?",
    "How to control aphids in wheat fields?"
]

print("=" * 80)
print("BASE MODEL vs FINE-TUNED MODEL COMPARISON")
print("=" * 80)

for question in test_questions:
    prompt = f"""### Instruction:\n{question}\n\n### Response:\n"""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(device)

    with torch.no_grad():
        base_output = base_model.generate(**inputs, max_new_tokens=100, do_sample=False)
        fine_output = model.generate(**inputs, max_new_tokens=100, do_sample=False)

    base_response = tokenizer.decode(base_output[0], skip_special_tokens=True)
    fine_response = tokenizer.decode(fine_output[0], skip_special_tokens=True)

    # Extract response part only
    if "### Response:" in base_response:
        base_response = base_response.split("### Response:")[-1].strip()
    if "### Response:" in fine_response:
        fine_response = fine_response.split("### Response:")[-1].strip()

    print(f"\nQuestion: {question}")
    print(f"\n  Base Model:  {base_response[:300]}")
    print(f"\n  Fine-Tuned:  {fine_response[:300]}")
    print("-" * 80)

# Free base model memory
del base_model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

## 11. Deployment with Gradio (UI Integration)

We deploy the fine-tuned model as an interactive chatbot using Gradio. Users can ask agriculture-related questions and receive domain-specific responses.

**Note:** On Google Colab, `share=True` creates a public URL accessible for 72 hours.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

def chatbot(user_input):
    """Generate a response to an agricultural question."""
    prompt = f"""### Instruction:\n{user_input}\n\n### Response:\n"""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=True,
            temperature=0.7
        )

    response = tokenizer.decode(output[0], skip_special_tokens=True)
    if "### Response:" in response:
        response = response.split("### Response:")[-1].strip()
    return response


# Create Gradio interface
interface = gr.Interface(
    fn=chatbot,
    inputs=gr.Textbox(
        lines=3,
        placeholder="Ask an agriculture question... (e.g., 'How do I improve soil fertility?')",
        label="Your Question"
    ),
    outputs=gr.Textbox(lines=5, label="AgriSmart Response"),
    title="AgriSmart Assistant",
    description="A domain-specific agricultural assistant fine-tuned on farming QA data. Ask questions about crops, pests, soil, fertilizers, and more.",
    examples=[
        ["How do I prevent cassava mosaic disease?"],
        ["What is the best fertilizer for rice crops?"],
        ["How to control aphids in wheat fields?"],
        ["What are the symptoms of nitrogen deficiency in maize?"],
    ],
)

interface.launch(share=True)